# Production RAG Pipeline

We've learned the pieces. Today, we put them together.

**PDF → Chunking → Vector DB → Retrieval → LLM** — a complete RAG system.

## Install PDF Library

In [2]:
%pip install PyPDF2 -q chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

In [3]:
from google import genai
import chromadb
from PyPDF2 import PdfReader
import os
from dotenv import load_dotenv

# from google.colab import userdata
# API_KEY = userdata.get('GEMINI_API_KEY')
load_dotenv(dotenv_path='./.env')
API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=API_KEY)

## Step 1: Load PDF

In [4]:
# Load PDF
pdf_path = "D:\AIML\gen_ai\leave_policy.pdf"  # Your PDF file
reader = PdfReader(pdf_path)

# Extract text from all pages
full_text = ""   # empty string
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(f"📄 Loaded PDF: {len(reader.pages)} pages, {len(full_text)} characters")
print(f"\n📖 Preview:\n{full_text[:500]}...")

📄 Loaded PDF: 8 pages, 19644 characters

📖 Preview:
 
 Pride Global India Employee Leave Policy  
PRIDE Technologies Consulting (India) Private Limited (Ver. 2.0) 
Russell Tobin Associates Staffing Solutions India Private Limited (Ver. 2.0) 
 
Date  Version  Rational change  Description  
15 January, 2025  1.0 Annual review  Periodic review of leave 
provisions and 
procedures to align 
with company and end 
client requirements  
6 March 2026  2.0 Annual review  Periodic review of leave 
provisions and 
procedures to align 
with company and end 
...


## Step 2: Chunk the Document

In [5]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = [] 
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()  # fixed chunking: 500 chars.
        if chunk:
            chunks.append(chunk)
        start = end - overlap   # 50 steps back  0-500  500-50 = 450
    return chunks

chunks = chunk_text(full_text)

print(f"✂️ Created {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i}: {chunk[:80]}...\n")

✂️ Created 44 chunks

Chunk 0: Pride Global India Employee Leave Policy  
PRIDE Technologies Consulting (India)...

Chunk 1: s and 
procedures to align 
with company and end 
client requirements  
 
 
  
V...

Chunk 2: ...........  3 
Public Holidays  ................................ .................



## Step 3: Store in Vector Database

In [7]:
# Create ChromaDB collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="pdf_store2")

# Generate embeddings and store
for i, chunk in enumerate(chunks):
    embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    ).embeddings[0].values

   # add chunks and embeddings in the chroma collection
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[f"chunk_{i}"]
    )

print(f"✅ Stored {collection.count()} chunks in ChromaDB")

✅ Stored 44 chunks in ChromaDB


## Step 4: The RAG Function

In [8]:
def ask(question):
    """Complete RAG pipeline: retrieve + generate."""

    # Embed the question
    query_emb = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    ).embeddings[0].values

    # Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=3
    )
    context = "\n\n".join(results['documents'][0])

    # Generate answer
    prompt = f"""Answer the question based ONLY on the context below.
If the answer is not in the context, say "I couldn't find that information."

Context:
{context}

Question: {question}

Answer:"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text, results['documents'][0]

## Test: Ask Questions About the PDF

In [9]:
question = "What is sick leave?"

answer, sources = ask(question)

print(f"❓ {question}\n")
print(f"💬 {answer}\n")
print("📄 Sources:")
for i, src in enumerate(sources[1:]):
    print(f"  [{i+1}] {src[:100]}...")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


❓ What is sick leave?

💬 Sick leave is leave allotted on a pro-rata basis based on the entitlement for each state in keeping with the Shops & Establishments Act, which should be used solely to address health and wellness concerns that require time-off and not for any other purpose.

📄 Sources:
  [1] ate for resuming duty after sickness, from a registered medical practitioner.  
Leave application (f...
  [2] be allowed to resume work only after 
the prescribed rest period. In order to resume his/her regular...


In [ ]:
# Try your own questions!
question = input("Ask a question: ")
answer, sources = ask(question)
print(f"\n💬 {answer}")

In [ ]:
# Try your own questions!
question = input("Ask a question: ")
answer, sources = ask(question)
print(f"\n💬 {answer}")

## The Complete Pipeline

```
PDF → Extract Text → Chunk → Embed → Store in Vector DB
                                        ↓
Question → Embed → Search → Retrieve → LLM → Answer
```

## Key Takeaways

1. **Extract** text from PDFs using PyPDF2
2. **Chunk** with overlap to preserve context
3. **Store** embeddings in a vector database
4. **Retrieve** and **Generate** — that's RAG

---

This wraps up RAG basics. **Next section: Agents** — when retrieval isn't enough and you need to take actions.

# **What is Hallucination**
# **How to reduce or handle the hallucination**
# **If ur model is not giving you the expected answer then how to handle this problem??**

# **If we have pdf with text, image, table, image of table then how to embed this data/handle this case???**

# **If we have pdfs cvs excel image files in GB's then how to handle the embedding of this????**